In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['CLV'] = df['MonthlyCharges'] * df['tenure']


In [11]:
# Q11: Annualized revenue loss due to churn
churned_df = df[df['Churn'] == 'Yes']

monthly_loss = churned_df['MonthlyCharges'].sum()
annual_loss = round(monthly_loss * 12, 2)

print(f'Annualized revenue loss: ${annual_loss:,.2f}')

Annualized revenue loss: $1,669,570.20


In [3]:
# Q12: Pareto analysis — Do 20% of telecom customers generate 80% of total revenue
df_pareto = df[df['TotalCharges'] > 0].copy()
df_pareto = df_pareto.sort_values('TotalCharges', ascending=False)

df_pareto['cumulative_revenue'] = df_pareto['TotalCharges'].cumsum()
df_pareto['cumulative_pct'] = df_pareto['cumulative_revenue'] / df_pareto['TotalCharges'].sum() * 100

top_20_pct_cutoff = int(len(df_pareto) * 0.20)
top_20_customers = df_pareto.head(top_20_pct_cutoff)

print(f'Total customers:{len(df_pareto)}')
print(f'Top 20% customer count:{top_20_pct_cutoff}')
print(f'Revenue from top 20%:${top_20_customers["TotalCharges"].sum():,.2f}')
print(f'Total revenue:${df_pareto["TotalCharges"].sum():,.2f}')
print(f'Top 20% revenue contribution: {round(top_20_customers["TotalCharges"].sum() / df_pareto["TotalCharges"].sum() * 100, 2)}%')

Total customers:              7032
Top 20% customer count:       1406
Revenue from top 20%:         $8,602,752.45
Total revenue:                $16,056,168.70
Top 20% revenue contribution: 53.58%


In [4]:
# Q13: ARPU by Contract Type, Internet Service, Payment Method
# ARPU stands for Average Revenue Per User, 
# which is calculated by dividing total revenue by the number of users. 
# In this case, we are calculating the average monthly charges for each contract type, 
# internet service, and payment method to understand how much revenue is generated on average from customers in each category.

arpu_contract = df.groupby('Contract')['MonthlyCharges'].mean().round(2).reset_index()
arpu_contract.columns = ['Contract', 'ARPU']
print("ARPU by Contract Type:")
print(arpu_contract.to_string(index=False))

print()
arpu_internet = df.groupby('InternetService')['MonthlyCharges'].mean().round(2).reset_index()
arpu_internet.columns = ['InternetService', 'ARPU']
print("ARPU by Internet Service:")
print(arpu_internet.to_string(index=False))

print()

arpu_payment = df.groupby('PaymentMethod')['MonthlyCharges'].mean().round(2).reset_index()
arpu_payment.columns = ['PaymentMethod', 'ARPU']
print("ARPU by Payment Method:")
print(arpu_payment.to_string(index=False))


ARPU by Contract Type:
      Contract  ARPU
Month-to-month 66.40
      One year 65.05
      Two year 60.77

ARPU by Internet Service:
InternetService  ARPU
            DSL 58.10
    Fiber optic 91.50
             No 21.08

ARPU by Payment Method:
            PaymentMethod  ARPU
Bank transfer (automatic) 67.19
  Credit card (automatic) 66.51
         Electronic check 76.26
             Mailed check 43.92


In [5]:
# Q14: Revenue leakage report — high charges, low tenure, churned
high_charge_threshold = df['MonthlyCharges'].mean()
low_tenure_threshold  = df['tenure'].quantile(0.25)

leakage = df[
    (df['MonthlyCharges'] > high_charge_threshold) &
    (df['tenure'] <= low_tenure_threshold) &
    (df['Churn'] == 'Yes')
][['customerID', 'Contract', 'InternetService', 'MonthlyCharges', 'tenure', 'TotalCharges']]

print(f'High charge threshold:${round(high_charge_threshold, 2)}')
print(f'Low tenure threshold:{int(low_tenure_threshold)} months')
print(f'Revenue leakage customers:{len(leakage)}')
print(f'Monthly revenue lost:${leakage["MonthlyCharges"].sum():,.2f}')
leakage.head(10)

High charge threshold:  $64.76
Low tenure threshold:   9 months
Revenue leakage customers: 577
Monthly revenue lost:   $47,006.95


,customerID,Contract,InternetService,MonthlyCharges,tenure,TotalCharges
4,9237-HQITU,Month-to-month,Fiber optic,70.70,2,151.65
5,9305-CDSKC,Month-to-month,Fiber optic,99.65,8,820.50
36,6047-YHPVI,Month-to-month,Fiber optic,69.70,5,316.90
47,7760-OYPDY,Month-to-month,Fiber optic,80.65,2,144.15
53,7495-OOKFY,Month-to-month,Fiber optic,80.65,8,633.30
64,5698-BQJOH,Month-to-month,Fiber optic,94.40,9,857.25
80,5919-TMRGD,Month-to-month,Fiber optic,79.35,1,79.35
82,9191-MYQKX,Month-to-month,Fiber optic,75.15,7,496.90
122,0404-SWRVG,Month-to-month,Fiber optic,74.40,3,229.55
139,0390-DCFDQ,Month-to-month,Fiber optic,70.45,1,70.45


In [6]:
# Q15: Customers paying more than 1.5 standard deviations above mean monthly charge
mean_charge = df['MonthlyCharges'].mean()
std_charge  = df['MonthlyCharges'].std()
threshold = mean_charge + 1.5 * std_charge

high_payers = df[df['MonthlyCharges'] > threshold][['customerID', 'MonthlyCharges', 'Contract', 'InternetService', 'Churn']]

print(f'Mean monthly charge:${round(mean_charge, 2)}')
print(f'Standard deviation:${round(std_charge, 2)}')
print(f'Threshold:${round(threshold, 2)}')
print(f'High paying customers:{len(high_payers)}')
print(f'Churn rate among them:{round((high_payers["Churn"] == "Yes").sum() / len(high_payers) * 100, 2)}%')
high_payers.head(10)

Mean monthly charge:     $64.76
Standard deviation:      $30.09
Threshold:               $109.9
High paying customers:   226
Churn rate among them:   13.72%


,customerID,MonthlyCharges,Contract,InternetService,Churn
15,3655-SNQYZ,113.25,Two year,Fiber optic,No
72,1891-QRQSA,111.60,Two year,Fiber optic,No
75,2673-CXQEU,110.50,One year,Fiber optic,No
93,6067-NGCEU,111.05,Month-to-month,Fiber optic,No
104,3192-NQECA,110.00,Two year,Fiber optic,Yes
197,6168-YBYNP,111.35,Month-to-month,Fiber optic,No
198,7255-SSFBC,112.25,Two year,Fiber optic,No
256,7017-VFHAY,115.10,Two year,Fiber optic,No
257,6655-LHBYW,114.35,One year,Fiber optic,No
264,1950-KSVVJ,113.30,One year,Fiber optic,No
